# Clinic activity D5 — readiness trên Kaggle shared envelope

Import vào **Kaggle CPU / Internet On**, rồi **Run All** (khoảng 40–70 phút). Source dirty đã nhúng và hash; không cần repo/commit/push, không chạy notebook cũ.

Notebook đo **profile môi trường mới** (CPU/affinity/cgroup/memory), dựng PostgreSQL16/RabbitMQ/Redis riêng, role không superuser theo component, build Linux + migrations, seed 100.000 lịch, rebuild V2, chạy đủ Core/Clinic/Notification + workers + relay + read worker V2 + Nginx; ghi ledger PID/vai trò, collector CPU, calibration generator 2× vào sink no-op và smoke S/C/P qua gateway.

**Không phải benchmark:** policy v9 chưa đóng băng (`performanceMeasurementsAllowed=false`); số thời gian là quan sát chức năng. Không claim cô lập tài nguyên. Secrets/URL/log ở `/tmp`, export chỉ allowlist.

In [ ]:
RUN_TRIAL = True
RUN_BENCHMARK = False
assert RUN_TRIAL is True and RUN_BENCHMARK is False, 'Notebook này chỉ hỗ trợ readiness D5.'
import base64, hashlib, importlib.util, io, json, os, platform, uuid
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
assert platform.system() == 'Linux' and Path('/kaggle/working').is_dir(), 'Chạy trong notebook Kaggle.'
assert os.geteuid() == 0, 'Setup native cần quyền root trong kernel trial.'

SOURCE_SHA256 = '7f9c98e8410a7709428cb0673ffe4155c87bbe293119a0463e29b72e0ea626a0'
SOURCE_BASE64 = '...'
raw = base64.b64decode(SOURCE_BASE64, validate=True)
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, 'Source archive hash mismatch'
with ZipFile(io.BytesIO(raw)) as z:
    validator = z.read('source/benchmarks/phase2/kaggle/make_activity_d5_notebook.py')
namespace = {'__name__': 'activity_d5_bundle_validator', '__file__': '/tmp/activity-d5-validator/benchmarks/phase2/kaggle/make_activity_d5_notebook.py'}
exec(compile(validator, '<verified-activity-d5-validator>', 'exec'), namespace)
source_manifest, values = namespace['verified_members'](raw, SOURCE_SHA256)
RUN = str(uuid.uuid4())
PRIVATE = Path('/tmp') / ('clinic-activity-d5-' + RUN)
OUTPUT = Path('/kaggle/working') / ('clinic-activity-d5-' + RUN)
PRIVATE.mkdir(mode=0o700); OUTPUT.mkdir(mode=0o700)
with (PRIVATE/'source.zip').open('xb') as handle:
    handle.write(raw)
ROOT = PRIVATE / 'source'; ROOT.mkdir(mode=0o700)
for name, value in values.items():
    destination = ROOT.joinpath(*PurePosixPath(name).parts)
    assert destination.resolve().is_relative_to(ROOT.resolve())
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('xb') as handle:
        handle.write(value)
spec = importlib.util.spec_from_file_location('activity_d5_native', ROOT/'benchmarks/phase2/kaggle/activity_d5_native.py')
runner = importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
print({'run': RUN, 'source_verified': True, 'source_archive_sha256': SOURCE_SHA256, 'benchmark': False})

## Chạy readiness
Mỗi stage in tiến độ. Lỗi thì dừng và vẫn export ZIP báo stage lỗi; log riêng giữ trong `/tmp`, không copy log/context/token vào chat.

In [ ]:
summary = runner.run_trial(ROOT, PRIVATE, OUTPUT, RUN, SOURCE_SHA256)
print(json.dumps({k: v for k, v in summary.items() if k != 'download'}, indent=2))
from IPython.display import display, FileLink
display(FileLink(summary['download']))

## Gửi ZIP kết quả
Tải file `.zip` ở link trên rồi gửi lại để review. Readiness chỉ là đầu vào cho việc đóng băng policy/profile; chưa có kết quả hiệu năng.